Part 1 - Python & Data Foundations: ingest, validate, clean, enrich

Task 1.1 — Read the file the foundations way

In [2]:
import csv
import json

# Read the original CSV file
with open("appointments.csv", "r", newline="") as file:

    reader = csv.DictReader(file)

    print("Headers:", reader.fieldnames)

    rows = list(reader)

    print("Total rows:", len(rows))


# Function to convert data types
def parse_row(row):

    converted = row.copy()

    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])

    return converted


# Separate accepted and rejected records
accepted = []
rejected = []

for row in rows:

    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)


# Save rejected records
with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)


# Display results
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("Rejected records saved successfully!")

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020
Accepted rows: 988
Rejected rows: 32
Rejected records saved successfully!


Task 1.2 — Inspect and clean, pass 1

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments.csv")

In [5]:
# Display the number of rows and columns
print("Shape:", df.shape)

# Display column information
df.info()

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())

Shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patient_id          0
clinic_id    

A. Remove exact duplicate rows

In [6]:
df = df.drop_duplicates()

print("After removing duplicates:", len(df))

After removing duplicates: 1000


Standardise booking_channel

In [7]:
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print(df["booking_channel"].value_counts())

booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64


C. Convert date columns

In [9]:
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)

D. Remove rows with missing appointment dates or status

In [12]:
df = df.dropna(subset=["appt_date", "status"])

print("Final shape:", df.shape)

Final shape: (990, 14)


Task 1.3 — Clean, pass 2: validity, gaps and outliers

Identifing invalid ages

In [13]:
# Replace impossible ages with missing values
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

print("Missing ages:", df["age"].isnull().sum())

Missing ages: 17


Filling missing values using median

In [14]:
# Find median values
print("Age median:", df["age"].median())
print("Distance median:", df["distance_km"].median())

# Fill missing values
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

# Check missing values
print("\nMissing values after filling:")
print(df[["age", "distance_km"]].isnull().sum())

Age median: 48.0
Distance median: 8.25

Missing values after filling:
age            0
distance_km    0
dtype: int64


Detect and cap outliers

In [15]:
# Calculate quartiles for distance
q1_distance = df["distance_km"].quantile(0.25)
q3_distance = df["distance_km"].quantile(0.75)

iqr_distance = q3_distance - q1_distance

lower_distance = q1_distance - 1.5 * iqr_distance
upper_distance = q3_distance + 1.5 * iqr_distance

# Count distance outliers
distance_capped = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
).sum()


# Calculate quartiles for fee
q1_fee = df["fee_inr"].quantile(0.25)
q3_fee = df["fee_inr"].quantile(0.75)

iqr_fee = q3_fee - q1_fee

lower_fee = q1_fee - 1.5 * iqr_fee
upper_fee = q3_fee + 1.5 * iqr_fee

# Count fee outliers
fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()


# Cap outliers using NumPy
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 26
Fee values capped: 3


Rounding the values

In [16]:
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

print(df[["age", "distance_km", "fee_inr"]].head())

   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700


Task 1.4 - Features and a NumPy summary

In [18]:
# Calculate days between booking and appointment
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

# Convert status into numerical values
df["no_show"] = (df["status"] == "No-Show").astype(int)

# Display results
print(df[["booked_on", "appt_date", "lead_days", "status", "no_show"]].head())

   booked_on  appt_date  lead_days    status  no_show
0 2025-01-24 2025-01-25          1   No-Show        1
1 2025-05-27 2025-06-06         10  Attended        0
2 2025-03-24 2025-03-30          6  Attended        0
4 2025-04-18 2025-04-22          4  Attended        0
5 2024-11-23 2025-01-02         40   No-Show        1


In [19]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

print(X.shape)

(990, 4)


In [20]:
column_means = np.mean(X, axis=0)

print("Column means:", column_means)

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [21]:
mask = df["lead_days"] > 14

print("Number of appointments:", mask.sum())
print("Share of appointments:", mask.mean())

Number of appointments: 296
Share of appointments: 0.298989898989899


In [22]:
distance = X[:, 1]

z_scores = (
    distance - np.mean(distance)
) / np.std(distance)

outlier_count = np.sum(np.abs(z_scores) > 2)

print("Rows with |z| > 2:", outlier_count)

Rows with |z| > 2: 53


Task 1.5 - Enrich clinic data using a live API

In [2]:
import requests
import pandas as pd

clinics_df = pd.read_csv("clinics.csv")

print(clinics_df.head())
print("Total clinics:", len(clinics_df))

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
Total clinics: 8


In [3]:
import requests

temperatures = []

for index, clinic in clinics_df.iterrows():

    latitude = clinic["latitude"]
    longitude = clinic["longitude"]

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp = None
    reason = None

    # Try twice if the API fails
    for attempt in range(2):

        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            data = response.json()

            hourly = data.get("hourly", {})
            values = hourly.get("temperature_2m")

            if not values:
                reason = "Hourly temperature data missing"
                continue

            temp = values[0]
            break

        except requests.exceptions.RequestException as e:
            reason = str(e)

    if temp is None:
        print(clinic["clinic_name"], ":", reason)

    temperatures.append(temp)

# Add temperature column
clinics_df["temp_c"] = temperatures

# Save enriched dataset
clinics_df.to_csv("clinics_enriched.csv", index=False)

print("\nEnriched clinic data:")
print(clinics_df)

print("\nTotal clinics:", len(clinics_df))
print("Temperature values collected:", clinics_df["temp_c"].notna().sum())

Kothrud Family Clinic : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Hadapsar Care Centre : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Sitabuldi Health Point : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Malviya Nagar Clinic : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Vaishali Nagar Clinic : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Edappally Medical Centre : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Vijay Nagar Clinic : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gomti Nagar Care Point : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)

Enriched clinic data:
  clinic_id               clinic_name     city   

Task 1.6 - Export the cleaned appointments

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments.csv")

print(df.shape)
print(df.head())

(1020, 14)
  appointment_id patient_id clinic_id   booked_on   appt_date   age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25  60.0      F   
1          A0002       P046       C07  2025-05-27  2025-06-06  39.0      M   
2          A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
3          A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
4          A0004       P482       C01  2025-04-18  2025-04-22  52.0      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-in     Follow-up            No   
2          4.1      850         Walk-in  Consultation            No   
3          4.1      850         Walk-in  Consultation            No   
4          7.5      750             App  Consultation           Yes   

   prior_no_shows    status  
0               1   No-Show  
1               2  Attended  
2  

In [9]:
# 1. Remove duplicate records
df = df.drop_duplicates()

# 2. Standardise booking channel
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

# 3. Convert date columns
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)
df["booked_on"] = pd.to_datetime(
    df["booked_on"], errors="coerce"
)

# 4. Remove rows with missing appointment date or status
df = df.dropna(subset=["appt_date", "status"])

# 5. Handle invalid ages
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

# 6. Fill missing distance
df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

# 7. Handle distance outliers using IQR
q1 = df["distance_km"].quantile(0.25)
q3 = df["distance_km"].quantile(0.75)
iqr = q3 - q1

df["distance_km"] = np.clip(
    df["distance_km"],
    q1 - 1.5 * iqr,
    q3 + 1.5 * iqr
)

# 8. Handle fee outliers using IQR
q1 = df["fee_inr"].quantile(0.25)
q3 = df["fee_inr"].quantile(0.75)
iqr = q3 - q1

df["fee_inr"] = np.clip(
    df["fee_inr"],
    q1 - 1.5 * iqr,
    q3 + 1.5 * iqr
)

# 9. Round values
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

# 10. Create features
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("Restored dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("No-show count:", df["no_show"].sum())

Restored dataset shape: (990, 16)
Missing values: 0
No-show count: 218


In [10]:
# Sort appointments
clean_export = df.copy()
clean_export = clean_export.sort_values("appointment_id")

# Format dates
clean_export["booked_on"] = pd.to_datetime(
    clean_export["booked_on"]
).dt.strftime("%Y-%m-%d")

clean_export["appt_date"] = pd.to_datetime(
    clean_export["appt_date"]
).dt.strftime("%Y-%m-%d")

# Required column order
columns = [
    "appointment_id", "patient_id", "clinic_id",
    "booked_on", "appt_date", "age", "gender",
    "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows",
    "status", "lead_days", "no_show"
]

clean_export = clean_export[columns]

# Validate
print("Dataset shape:", clean_export.shape)
print("Unique appointment IDs:",
      clean_export["appointment_id"].nunique())
print("Missing values:", clean_export.isnull().sum().sum())

# Save
clean_export.to_csv("appointments_clean.csv", index=False)

print("appointments_clean.csv saved!")

Dataset shape: (990, 16)
Unique appointment IDs: 990
Missing values: 0
appointments_clean.csv saved!
